# Batch Fraud Detection Model Training (Spark ML)

## Objective

The objective of this notebook is to train a machine learning model using the preprocessed PaySim transaction dataset.

In this notebook:

- Load prepared training and testing datasets
- Train a Random Forest classification model
- Generate predictions
- Evaluate model performance
- Save the trained Spark ML model

In [1]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

from pyspark.ml.feature import (
    StringIndexer,
    VectorAssembler
)

from pyspark.ml.classification import RandomForestClassifier

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

from pyspark.ml import Pipeline

## Spark Session Setup

Initializes Apache Spark and configures the environment required for model training.

In [2]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"


spark = SparkSession.builder \
    .appName("FraudModelTraining") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()


print("Spark Version:", spark.version)

Spark Version: 3.5.9


## Load Original Dataset

Load the PaySim dataset that will be used for preprocessing and model training.

In [3]:
from pathlib import Path

project_root = Path(
    r"C:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline"
)

csv_path = project_root / "data" / "PS_20174392719_1491204439457_log.csv"

fraud_df = spark.read.csv(
    str(csv_path),
    header=True,
    inferSchema=True
)

print("Dataset Loaded Successfully")

print("Rows:", fraud_df.count())
print("Columns:", len(fraud_df.columns))

Dataset Loaded Successfully
Rows: 6362620
Columns: 11


## Remove Unnecessary Columns

The columns `nameOrig` and `nameDest` are unique transaction identifiers and do not contribute to fraud prediction.

The `isFlaggedFraud` column is removed because it is a system-generated flag that may introduce bias into the model.

In [4]:
processed_df = fraud_df.drop(
    "nameOrig",
    "nameDest",
    "isFlaggedFraud"
)

print("Remaining Columns:")
processed_df.columns

Remaining Columns:


['step',
 'type',
 'amount',
 'oldbalanceOrg',
 'newbalanceOrig',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud']

## Feature Engineering

Two additional features are created to capture transaction behaviour.

- **orig_balance_change**: Difference between the sender's balance before and after the transaction.
- **dest_balance_change**: Difference between the receiver's balance after and before the transaction.

These engineered features help the model detect suspicious money movement patterns.

In [5]:
processed_df = processed_df.withColumn(
    "orig_balance_change",
    col("oldbalanceOrg") - col("newbalanceOrig")
)

processed_df = processed_df.withColumn(
    "dest_balance_change",
    col("newbalanceDest") - col("oldbalanceDest")
)

processed_df.show(5)

+----+--------+--------+-------------+--------------+--------------+--------------+-------+-------------------+-------------------+
|step|    type|  amount|oldbalanceOrg|newbalanceOrig|oldbalanceDest|newbalanceDest|isFraud|orig_balance_change|dest_balance_change|
+----+--------+--------+-------------+--------------+--------------+--------------+-------+-------------------+-------------------+
|   1| PAYMENT| 9839.64|     170136.0|     160296.36|           0.0|           0.0|      0|  9839.640000000014|                0.0|
|   1| PAYMENT| 1864.28|      21249.0|      19384.72|           0.0|           0.0|      0| 1864.2799999999988|                0.0|
|   1|TRANSFER|   181.0|        181.0|           0.0|           0.0|           0.0|      1|              181.0|                0.0|
|   1|CASH_OUT|   181.0|        181.0|           0.0|       21182.0|           0.0|      1|              181.0|           -21182.0|
|   1| PAYMENT|11668.14|      41554.0|      29885.86|           0.0|        

## Encode Transaction Type

The `type` column contains categorical transaction types such as `PAYMENT`, `TRANSFER`, and `CASH_OUT`.

Spark's `StringIndexer` converts these categories into numerical values required for machine learning.

In [6]:
from pyspark.ml.feature import StringIndexer

type_indexer = StringIndexer(
    inputCol="type",
    outputCol="type_index"
)

processed_df = type_indexer.fit(processed_df).transform(processed_df)

processed_df.select(
    "type",
    "type_index"
).show(10)

+--------+----------+
|    type|type_index|
+--------+----------+
| PAYMENT|       1.0|
| PAYMENT|       1.0|
|TRANSFER|       3.0|
|CASH_OUT|       0.0|
| PAYMENT|       1.0|
| PAYMENT|       1.0|
| PAYMENT|       1.0|
| PAYMENT|       1.0|
| PAYMENT|       1.0|
|   DEBIT|       4.0|
+--------+----------+
only showing top 10 rows



## Create Feature Vector

All numerical features are combined into a single `features` column using `VectorAssembler`.

This feature vector serves as the input for the Spark ML Random Forest model.

In [7]:
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "orig_balance_change",
    "dest_balance_change",
    "type_index"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

final_df = assembler.transform(processed_df)

final_df.select(
    "features",
    "isFraud"
).show(5, truncate=False)

+------------------------------------------------------------------+-------+
|features                                                          |isFraud|
+------------------------------------------------------------------+-------+
|[1.0,9839.64,170136.0,160296.36,0.0,0.0,9839.640000000014,0.0,1.0]|0      |
|[1.0,1864.28,21249.0,19384.72,0.0,0.0,1864.2799999999988,0.0,1.0] |0      |
|[1.0,181.0,181.0,0.0,0.0,0.0,181.0,0.0,3.0]                       |1      |
|[1.0,181.0,181.0,0.0,21182.0,0.0,181.0,-21182.0,0.0]              |1      |
|[1.0,11668.14,41554.0,29885.86,0.0,0.0,11668.14,0.0,1.0]          |0      |
+------------------------------------------------------------------+-------+
only showing top 5 rows



## Prepare Dataset for Model Training

The processed dataset is reduced to two columns:

- **features** → Input features for the machine learning model.
- **label** → Target variable indicating fraudulent (`1`) or legitimate (`0`) transactions.

In [8]:
model_df = final_df.select(
    "features",
    col("isFraud").alias("label")
)

model_df.show(5)

model_df.printSchema()

+--------------------+-----+
|            features|label|
+--------------------+-----+
|[1.0,9839.64,1701...|    0|
|[1.0,1864.28,2124...|    0|
|[1.0,181.0,181.0,...|    1|
|[1.0,181.0,181.0,...|    1|
|[1.0,11668.14,415...|    0|
+--------------------+-----+
only showing top 5 rows

root
 |-- features: vector (nullable = true)
 |-- label: integer (nullable = true)



## Split Dataset

The dataset is divided into training and testing datasets.

- **80%** for training
- **20%** for testing

This allows the model to be evaluated on unseen data.

In [9]:
train_df, test_df = model_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training Records:", train_df.count())
print("Testing Records :", test_df.count())

Training Records: 5089858
Testing Records : 1272762


## Fraud Distribution

Verify that both the training and testing datasets contain fraudulent and legitimate transactions.

In [10]:
print("Training Dataset")

train_df.groupBy("label") \
    .count() \
    .show()

print("Testing Dataset")

test_df.groupBy("label") \
    .count() \
    .show()

Training Dataset
+-----+-------+
|label|  count|
+-----+-------+
|    0|5083312|
|    1|   6546|
+-----+-------+

Testing Dataset
+-----+-------+
|label|  count|
+-----+-------+
|    0|1271095|
|    1|   1667|
+-----+-------+



## Train Random Forest Model

A Random Forest Classifier is trained using the prepared feature vectors.

Random Forest combines multiple decision trees to improve prediction accuracy and reduce overfitting.

In [11]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    numTrees=100,
    seed=42
)

model = rf.fit(train_df)

print("Random Forest Model Trained Successfully!")

Random Forest Model Trained Successfully!


## Generate Predictions

The trained model is applied to the testing dataset to predict whether each transaction is fraudulent or legitimate.

In [12]:
predictions = model.transform(test_df)

predictions.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----+----------+-----------------------------------------+
|label|prediction|probability                              |
+-----+----------+-----------------------------------------+
|0    |0.0       |[0.9917663873942696,0.008233612605730375]|
|0    |0.0       |[0.9917663873942696,0.008233612605730375]|
|0    |0.0       |[0.9917763829258459,0.008223617074154164]|
|0    |0.0       |[0.9917763829258459,0.008223617074154164]|
|0    |0.0       |[0.9917449998529009,0.008255000147099211]|
|0    |0.0       |[0.9833547646376333,0.016645235362366697]|
|0    |0.0       |[0.9917619012904427,0.008238098709557261]|
|0    |0.0       |[0.9917663873942696,0.008233612605730375]|
|0    |0.0       |[0.9917477770664597,0.008252222933540321]|
|0    |0.0       |[0.9917477770664597,0.008252222933540321]|
+-----+----------+-----------------------------------------+
only showing top 10 rows



## Model Evaluation

The trained Random Forest model is evaluated using the testing dataset.

Model accuracy is calculated to measure how well the model classifies fraudulent and legitimate transactions.

In [13]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.9993392323152326


## Prediction Summary

The following table compares the actual fraud labels with the model's predicted labels.

This provides a quick overview of the prediction results.

In [14]:
predictions.groupBy(
    "label",
    "prediction"
).count().show()

+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|    1|       1.0|    833|
|    0|       0.0|1271088|
|    1|       0.0|    834|
|    0|       1.0|      7|
+-----+----------+-------+



# Trained and evaluated the Spark ML Random Forest model

In [17]:
print("Spark Random Forest model trained successfully.")
print("Accuracy:", accuracy)

print("Model is ready for deployment in the real-time fraud detection pipeline.")

spark.stop()

print("Spark session stopped successfully.")

Spark Random Forest model trained successfully.
Accuracy: 0.9993392323152326
Model is ready for deployment in the real-time fraud detection pipeline.
Spark session stopped successfully.


# Summary

## Completed Tasks

- Loaded the PaySim transaction dataset
- Removed unnecessary identifier columns
- Engineered additional balance-related features
- Encoded transaction types using StringIndexer
- Created feature vectors for Spark ML
- Split the dataset into training and testing sets
- Trained a Random Forest classifier
- Generated fraud predictions
- Evaluated model accuracy
- Trained and evaluated the Spark ML Random Forest model

The trained model is now ready for integration into the real-time fraud detection pipeline.